# One-Click Colab: Soft-Attention + Ablations

This notebook:
- clones the repo from GitHub (branch `soft-attention-ablations`)
- prepares Flickr8k under `data/flickr8k` (uses the repo copy if present; otherwise downloads the public release)
- saves persistent outputs to Google Drive:
  - baseline: `MyDrive/Rohan_DL_Project_Outputs/{checkpoints,results}`
  - ablations: `MyDrive/Rohan_DL_Project_Outputs/ablationExperiments/`

Run on a GPU runtime for training.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Miwi343/Neural_Image_Caption_Generator.git'
BRANCH = 'soft-attention-ablations'
REPO_DIR = Path('/content/Neural_Image_Caption_Generator')
BASE_OUTPUT_DIR = Path('/content/drive/MyDrive/Rohan_DL_Project_Outputs')
ABLATIONS_OUTPUT_DIR = Path('/content/drive/MyDrive/Rohan_DL_Project_Outputs/ablationExperiments')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    REPO_URL,
    str(REPO_DIR),
], check=True)

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ABLATIONS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _symlink_dir(local: Path, target: Path) -> None:
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink() or local.is_file():
        local.unlink()
    elif local.exists():
        shutil.rmtree(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    local.symlink_to(target, target_is_directory=True)


# Baseline outputs (kept identical to existing baseline defaults)
_symlink_dir(REPO_DIR / 'checkpoints', BASE_OUTPUT_DIR / 'checkpoints')
_symlink_dir(REPO_DIR / 'results',     BASE_OUTPUT_DIR / 'results')

# Ablation outputs (scripts/run_ablations.py writes to outputs/ablations/...)
(REPO_DIR / 'outputs').mkdir(parents=True, exist_ok=True)
_symlink_dir(REPO_DIR / 'outputs' / 'ablations', ABLATIONS_OUTPUT_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print('Repo:', REPO_DIR)
print('Baseline outputs:', BASE_OUTPUT_DIR)
print('Ablation outputs:', ABLATIONS_OUTPUT_DIR)


## Install Dependencies

In [ ]:
!python -m pip install -q -r requirements.txt

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## Prepare And Validate Flickr8k

In [ ]:
import subprocess
import sys

from utils import validate_dataset_layout

DATA_ROOT = REPO_DIR / 'data' / 'flickr8k'
assert DATA_ROOT.exists(), f'Dataset folder missing from cloned repo: {DATA_ROOT}'

setup_cmd = [
    sys.executable,
    'scripts/colab_setup.py',
    '--repo_dir', str(REPO_DIR),
    '--data_root', 'data/flickr8k',
    '--source_dir', str(DATA_ROOT),
    '--require_official_splits',
]

try:
    subprocess.run(setup_cmd, check=True)
except subprocess.CalledProcessError:
    print('Repo data/flickr8k is incomplete; downloading the public Flickr8k release into data/flickr8k.')
    subprocess.run([
        sys.executable,
        'scripts/colab_setup.py',
        '--repo_dir', str(REPO_DIR),
        '--data_root', 'data/flickr8k',
        '--download_public_flickr8k',
        '--require_official_splits',
    ], check=True)

validate_dataset_layout(str(DATA_ROOT), strict_split_counts=True)
print('Flickr8k validated:', DATA_ROOT)


## Run Tests

In [ ]:
!pytest -q


## Train Baseline (Soft Attention)

In [ ]:
# Baseline training writes to Drive-backed checkpoints/ + results/
!python train.py


## Evaluate Baseline

In [ ]:
!python evaluate.py   --checkpoint checkpoints/best.pt   --data_root data/flickr8k   --vocab data/flickr8k/vocab.json   --split val   --beam_width 1   --batch_size 64   --results_out results/val_eval.json

!python evaluate.py   --checkpoint checkpoints/best.pt   --data_root data/flickr8k   --vocab data/flickr8k/vocab.json   --split test   --beam_width 1   --batch_size 64   --results_out results/test_eval.json


## Visualize Baseline Attention

In [ ]:
!python visualize.py   --checkpoint checkpoints/best.pt   --vocab data/flickr8k/vocab.json   --data_root data/flickr8k   --split test   --num_examples 6   --output_dir results/attention_examples   --overlay_style paper   --dpi 200


## Run All Ablation Experiments

In [ ]:
# This runs *all* defined ablations in scripts/run_ablations.py.
# It is expensive; use --resume to skip completed runs.
!python scripts/run_ablations.py --run --resume


## Ablation Summary Table

In [ ]:
!python scripts/run_ablations.py --summary


## Output Locations

In [ ]:
from pathlib import Path

print('Baseline outputs:')
print(Path('/content/drive/MyDrive/Rohan_DL_Project_Outputs'))
print('Ablation outputs:')
print(Path('/content/drive/MyDrive/Rohan_DL_Project_Outputs/ablationExperiments'))
